# 案例一：幫領養站整理貓狗照片

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/ai_solution_practicum/01_cnn_pet_story.ipynb)

領養站收到一批照片，希望先把貓與狗分成兩區，再由工作人員
複核。你的任務不是取代工作人員，而是先把明顯案例排好，
讓人工把時間留給模糊照片。

**情境問題：** 加入水平翻轉後，驗證集 F1 會不會改善？

- baseline：小型 CNN，不做資料增強。
- candidate：模型與訓練設定都相同，只加入水平翻轉。
- 驗證邊界：兩個版本使用同一批訓練與驗證資料。
- [Kaggle 題目出處：Cats and Dogs image classification]
  (https://www.kaggle.com/datasets/samuelcortinhas/cats-and-dogs-image-classification)

實際練習資料採 **CIFAR-10 的 cat（類別 3）與 dog（類別 5）子集**，
由 Keras 在 Colab 內直接下載，不需要開啟上方網站。


## 1. 設定固定種子與快速模式


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

SEED = 20260719
QUICK_MODE = True
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
TRAIN_PER_CLASS = 1_200 if QUICK_MODE else None
VALIDATION_PER_CLASS = 400 if QUICK_MODE else None
EPOCHS = 3 if QUICK_MODE else 8
CAT_CLASS = 3
DOG_CLASS = 5

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("運算裝置:", tf.config.list_physical_devices("GPU") or "CPU")


## 2. 由 Keras 直接取得 CIFAR-10

Keras 會下載 CIFAR-10 官方資料。這裡只保留 cat 與 dog，
並沿用官方 train／test 邊界；test 在本練習中只當作 validation，
不參與模型訓練或設定調整。


In [ ]:
(all_train_images, all_train_labels), (
    all_validation_images,
    all_validation_labels,
) = tf.keras.datasets.cifar10.load_data()

class_names = ["cat", "dog"]


def select_cat_dog(
    images: np.ndarray,
    labels: np.ndarray,
    per_class: int | None,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    labels = labels.ravel()
    rng = np.random.default_rng(seed)
    selected_indices = []

    for original_class in (CAT_CLASS, DOG_CLASS):
        class_indices = np.flatnonzero(labels == original_class)
        rng.shuffle(class_indices)
        if per_class is not None:
            class_indices = class_indices[:per_class]
        selected_indices.append(class_indices)

    selected_indices = np.concatenate(selected_indices)
    rng.shuffle(selected_indices)
    binary_labels = (labels[selected_indices] == DOG_CLASS).astype(np.float32)
    return images[selected_indices], binary_labels


train_images, train_labels = select_cat_dog(
    all_train_images,
    all_train_labels,
    TRAIN_PER_CLASS,
    SEED,
)
validation_images, validation_labels_array = select_cat_dog(
    all_validation_images,
    all_validation_labels,
    VALIDATION_PER_CLASS,
    SEED + 1,
)

assert set(np.unique(train_labels)) == {0.0, 1.0}
assert set(np.unique(validation_labels_array)) == {0.0, 1.0}
print("訓練／驗證筆數:", len(train_images), len(validation_images))
print("二元標籤:", dict(enumerate(class_names)))


## 3. 小量資料健全性檢查


In [ ]:
figure, axes = plt.subplots(2, 4, figsize=(10, 5))
for image, label, axis in zip(
    train_images[:8],
    train_labels[:8],
    axes.ravel(),
):
    axis.imshow(image)
    axis.set_title(class_names[int(label)])
    axis.axis("off")
plt.suptitle("先看資料：CIFAR-10 的 32×32 貓狗影像")
plt.tight_layout()
plt.show()

def resize_image(image, label):
    return tf.image.resize(image, IMAGE_SIZE), label


train_ds = tf.data.Dataset.from_tensor_slices(
    (train_images, train_labels)
)
train_ds = train_ds.shuffle(
    len(train_images),
    seed=SEED,
    reshuffle_each_iteration=False,
).map(
    resize_image,
    num_parallel_calls=tf.data.AUTOTUNE,
).batch(BATCH_SIZE)

validation_ds = tf.data.Dataset.from_tensor_slices(
    (validation_images, validation_labels_array)
)
validation_ds = validation_ds.map(
    resize_image,
    num_parallel_calls=tf.data.AUTOTUNE,
).batch(BATCH_SIZE)

autotune = tf.data.AUTOTUNE
if QUICK_MODE:
    train_ds = train_ds.cache()
    validation_ds = validation_ds.cache()
train_ds = train_ds.prefetch(autotune)
validation_ds = validation_ds.prefetch(autotune)


## 4. 建立可公平比較的 CNN

兩個版本只差 use_flip。層數、批次、epoch、optimizer 與資料切分
全部不變。


In [ ]:
def build_cnn(use_flip: bool) -> tf.keras.Model:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    layers = [
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Rescaling(1.0 / 255),
    ]
    if use_flip:
        layers.append(tf.keras.layers.RandomFlip("horizontal", seed=SEED))
    layers.extend(
        [
            tf.keras.layers.Conv2D(16, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(64, 3, activation="relu"),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dropout(0.2, seed=SEED),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model = tf.keras.Sequential(layers)
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def evaluate_binary_model(model, dataset) -> tuple[dict, np.ndarray, np.ndarray]:
    probabilities = model.predict(dataset, verbose=0).ravel()
    labels = np.concatenate([batch_labels.numpy().ravel() for _, batch_labels in dataset])
    predictions = (probabilities >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
    }
    return metrics, probabilities, labels.astype(int)


## 5. baseline：不做水平翻轉


In [ ]:
baseline_model = build_cnn(use_flip=False)
baseline_history = baseline_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    verbose=2,
)
baseline_metrics, baseline_probabilities, validation_labels = evaluate_binary_model(
    baseline_model,
    validation_ds,
)
baseline_metrics


## 6. candidate：只加入水平翻轉


In [ ]:
candidate_model = build_cnn(use_flip=True)
candidate_history = candidate_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    verbose=2,
)
candidate_metrics, candidate_probabilities, candidate_labels = evaluate_binary_model(
    candidate_model,
    validation_ds,
)
assert np.array_equal(validation_labels, candidate_labels)
candidate_metrics


## 7. 比較前後證據

資料增強不保證一定進步。若 candidate 下降，先記錄結果，
不要重跑到出現喜歡的數字。


In [ ]:
comparison = pd.DataFrame(
    [baseline_metrics, candidate_metrics],
    index=["baseline_no_flip", "candidate_horizontal_flip"],
)
display(comparison.style.format("{:.3f}"))

f1_change = candidate_metrics["f1"] - baseline_metrics["f1"]
print(f"F1 改變：{f1_change:+.3f}")
chosen_version = "candidate" if f1_change > 0 else "baseline"
print("依本次驗證 F1 暫選：", chosen_version)


## 8. 看失敗案例，不只看總分


In [ ]:
predictions = (candidate_probabilities >= 0.5).astype(int)
error_indices = np.flatnonzero(predictions != validation_labels)

if len(error_indices):
    selected = error_indices[:6]
    figure, axes = plt.subplots(2, 3, figsize=(10, 7))
    for index, axis in zip(selected, axes.ravel()):
        axis.imshow(validation_images[index])
        axis.set_title(
            f"真實 {class_names[validation_labels[index]]}\n"
            f"預測 {class_names[predictions[index]]} "
            f"p={candidate_probabilities[index]:.2f}"
        )
        axis.axis("off")
    for axis in axes.ravel()[len(selected):]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("本次抽樣沒有錯誤；完整模式仍要再檢查。")


## 9. 限制與人工介入

- CIFAR-10 原圖只有 32×32；放大方便模型運算，不會增加原本不存在的細節。
- 這是小量、二分類教學子集，不能代表真實領養站的品種與拍攝條件。
- F1 是驗證集表現，不是上線後的保證。
- 遮擋、多人多寵物、極暗照片與非貓狗照片需要人工複核。
- 實務上可把接近 0.5 的案例送入「待人工確認」，不要硬分。

請補一句結論：你會採用哪一版？什麼照片一定交給人判斷？


## 10. 下載實驗紀錄


In [ ]:
experiment_record = {
    "case": "cnn_pet_story",
    "question": "加入水平翻轉後，驗證集 F1 是否改善？",
    "baseline": {"change": "不做資料增強", "metrics": baseline_metrics},
    "candidate": {"change": "只加入水平翻轉", "metrics": candidate_metrics},
    "single_change": "RandomFlip(horizontal)",
    "selected_by_validation_f1": chosen_version,
    "f1_change": float(f1_change),
    "human_review": "模糊、非貓狗、遮擋或機率接近 0.5 的照片",
    "limitation": "教學子集，不代表真實場域分布",
}

record_path = Path("/content/cnn_pet_experiment.json")
record_path.write_text(
    json.dumps(experiment_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(record_path)


In [ ]:
try:
    from google.colab import files
    files.download(str(record_path))
except ImportError:
    print("檔案已保留在", record_path)
